# %%
# =========================================
# STEP 1: Overview / Script Description (UPDATED)
# =========================================
# This script processes EPC-linked building simulation results to generate a
# detailed battery operation dataset for each dwelling.
#
# For each building:
# - Reads energy_results.csv from its TOTAL folder
# - Extracts electricity demand and PV generation
# - Handles cases where PV is missing by using existing_pv
#
# For each timestep (half-hourly, 17520 steps), it computes in order:
# 1. PV consumed and PV surplus
# 2. Battery charge (based on surplus and constraints)
# 3. Battery discharge (based on demand deficit and constraints)
# 4. Battery level update (state of charge tracking)
# 5. Battery export (remaining surplus after charging)
# 6. Net electricity demand (electricity_tot_9)
#
# Outputs:
# - battery.csv saved in each building's TOTAL folder
# - Includes full battery dynamics and resulting grid demand

In [1]:
# %%
# =========================================
# STEP 2: Import Libraries
# =========================================
import pandas as pd
import numpy as np
import os

In [2]:
# %%
# =========================================
# STEP 3: Define File Paths
# =========================================
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_lifetimes_2.csv"
baseline_folder = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [3]:
# %%
# =========================================
# STEP 4: Define Battery Parameters
# =========================================
battery_capacity = 10000
usable_capacity_factor = 0.9
usable_battery_capacity = battery_capacity * usable_capacity_factor

charge_eff = 0.92
discharge_eff = 0.92

half_hour_max_charge_energy = 1500
half_hour_max_discharge_energy = 1500

In [4]:
# %%
# =========================================
# STEP 5: Load EPC Dataset
# =========================================
epc_df = pd.read_csv(epc_file)

building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str).tolist()

In [5]:
# %%
# =========================================
# STEP 6: Create Half-Hourly Time Column
# =========================================
time_index = pd.date_range(start="00:30:00", periods=17520, freq="30min")
time_strings = time_index.strftime("%H:%M:%S")

In [6]:
# %%
# =========================================
# STEP 7: Process Each Building (UPDATED WITH electricity_tot_9)
# =========================================
for building_id in building_ids:
    
    total_folder = os.path.join(baseline_folder, building_id, "TOTAL")
    input_file = os.path.join(total_folder, "energy_results.csv")
    output_file = os.path.join(total_folder, "battery.csv")
    
    if not os.path.exists(input_file):
        print(f"Missing file for building {building_id}")
        continue
    
    try:
        df = pd.read_csv(input_file)
        
        required_cols = ["electricity_tot_7", "pv", "existing_pv"]
        if not all(col in df.columns for col in required_cols):
            print(f"Missing columns in {building_id}")
            continue
        
        electricity = df["electricity_tot_7"].values
        pv = df["pv"].values
        
        if np.sum(pv) == 0:
            pv = df["existing_pv"].values
        
        n = len(electricity)
        
        # Initialise arrays
        pv_consumed = np.zeros(n)
        pv_surplus = np.zeros(n)
        battery_charge = np.zeros(n)
        battery_discharge = np.zeros(n)
        battery_level = np.zeros(n)
        battery_export = np.zeros(n)
        electricity_tot_9 = np.zeros(n)
        
        prev_battery_level = 0
        
        for t in range(n):
            
            # ---------------------------------
            # 1. PV consumed & surplus
            # ---------------------------------
            pv_consumed[t] = min(electricity[t], pv[t])
            pv_surplus[t] = max(pv[t] - electricity[t], 0)
            
            # ---------------------------------
            # 2. Battery charge
            # ---------------------------------
            battery_charge[t] = min(
                pv_surplus[t],
                half_hour_max_charge_energy,
                usable_battery_capacity - prev_battery_level
            )
            
            # ---------------------------------
            # 3. Battery discharge
            # ---------------------------------
            battery_discharge[t] = min(
                max(0, electricity[t] - pv[t]),
                half_hour_max_discharge_energy,
                prev_battery_level
            )
            
            # ---------------------------------
            # 4. Battery level update
            # ---------------------------------
            battery_level[t] = min(
                usable_battery_capacity,
                prev_battery_level 
                + (battery_charge[t] * charge_eff) 
                - (battery_discharge[t] / discharge_eff)
            )
            
            # ---------------------------------
            # 5. Battery export
            # ---------------------------------
            battery_export[t] = max(0, pv_surplus[t] - battery_charge[t])
            
            # ---------------------------------
            # 6. Net electricity demand
            # ---------------------------------
            electricity_tot_9[t] = max(
                0,
                electricity[t] - pv_consumed[t] - battery_discharge[t]
            )
            
            # Update state
            prev_battery_level = battery_level[t]
        
        # Output DataFrame
        out_df = pd.DataFrame({
            "Time": time_strings,
            "electricity_tot_7": electricity,
            "pv": pv,
            "pv_consumed": pv_consumed,
            "pv_surplus": pv_surplus,
            "battery_charge": battery_charge,
            "battery_level": battery_level,
            "battery_discharge": battery_discharge,
            "battery_export": battery_export,
            "electricity_tot_9": electricity_tot_9
        })
        
        out_df.to_csv(output_file, index=False)
        
        print(f"Processed building {building_id}")
    
    except Exception as e:
        print(f"Error processing {building_id}: {e}")

Processed building 1001829067
Processed building 1001951858
Processed building 1001829071
Processed building 1234761001
Processed building 1001991633
Processed building 1001664929
Processed building 1001829059
Processed building 1001063639
Processed building 1234761000
Processed building 1236594950
Processed building 1001664925
Processed building 1001906271
Processed building 1000238911
Processed building 1001829057
Processed building 1234760999
Processed building 1002143357
Processed building 1001951854
Processed building 1001829069
Processed building 1002313096
Processed building 1002143351
Processed building 1001870854
Processed building 1001870864
Processed building 1002143293
Processed building 1002143352
Processed building 1002143353
Processed building 1234647955
Processed building 1002313093
Processed building 1001991629
Processed building 1001991628
Processed building 1000238920
Processed building 1001829085
Processed building 1001063646
Processed building 1001829058
Processed 

In [7]:
# %%
# =========================================
# STEP 8: Completion Message
# =========================================
print("All buildings processed. Battery CSV files created.")

All buildings processed. Battery CSV files created.
